In [1]:
# trims square area to actual basin using graphs
# change input and output names and lat and lon of point that you want to delineate to

import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.geometry import Point
import fiona

# -----------------------------
# USER INPUTS
# -----------------------------
streams_fp = "merged_basins/saskTotalStreams.shp"
basins_fp = "merged_basins/saskTotalBasins.shp"  # must share LINKNO or same ID
output_streams_fp = "final_basin/Sask_streams.shp"
output_basins_fp = "final_basin/Sask_basins.shp"

# Input point in WGS84 (longitude, latitude)
lon, lat = -99.25689, 53.1854

# Field names
link_field = "LINKNO"
dslink_field = "DSLINKNO"
basin_link_field = "DN"

# -----------------------------
# EXPORT HELPER (FIONA SCHEMA ENFORCEMENT)
# -----------------------------
def export_fixed(gdf, filename):
    """Programmatically widens Shapefile schema field limits to prevent OGR truncation errors."""
    export_gdf = gdf.copy()
    
    # 1. Broaden floats and decimals
    float_cols = export_gdf.select_dtypes(include=['float64', 'float32']).columns
    for col in float_cols:
        if col != 'Slope':  
            export_gdf[col] = pd.to_numeric(export_gdf[col], errors='coerce').fillna(0.0).round(3)
            
    if 'Slope' in export_gdf.columns:
        export_gdf['Slope'] = pd.to_numeric(export_gdf['Slope'], errors='coerce').fillna(0.0)
    
    # 2. Prevent integer tracking decay to float representations
    int_cols = export_gdf.select_dtypes(include=['int64', 'int32']).columns
    for col in int_cols:
        export_gdf[col] = export_gdf[col].fillna(-1).astype(int)

    # 3. Intercept and re-write structural maximum widths
    schema = gpd.io.file.infer_schema(export_gdf)
    for col in float_cols:
        if col in schema['properties'] and col != 'Slope':
            schema['properties'][col] = 'float:24.3'  # Expanded width prevents overflow on large accumulation fields
    for col in int_cols:
        if col in schema['properties']:
            schema['properties'][col] = 'int:18'      # Wide bounds for long ID fields

    try:
        export_gdf.to_file(filename, driver="ESRI Shapefile", schema=schema, engine="fiona")
    except Exception as e:
        print(f"Fiona export optimization failed for {filename}, executing string fallback. Error: {e}")
        for col in list(float_cols) + list(int_cols):
            if col in export_gdf.columns: 
                export_gdf[col] = export_gdf[col].astype(str)
        export_gdf.to_file(filename, driver="ESRI Shapefile")

# -----------------------------
# LOAD DATA
# -----------------------------
print("Loading geofabric layers...")
streams = gpd.read_file(streams_fp)
basins = gpd.read_file(basins_fp)

# Ensure CRS match
if basins.crs != streams.crs:
    basins = basins.to_crs(streams.crs)

print(f"Streams CRS: {streams.crs}")

# -----------------------------
# BUILD GRAPH (downstream)
# -----------------------------
G = nx.DiGraph()

for _, row in streams.iterrows():
    link = row[link_field]
    dslink = row[dslink_field]

    if dslink not in [None, -1]:
        G.add_edge(link, dslink)

# Reverse graph for upstream traversal
G_rev = G.reverse()

# -----------------------------
# CREATE POINT (WGS84 → streams CRS)
# -----------------------------
point_wgs84 = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326")
point_proj = point_wgs84.to_crs(streams.crs)[0]

# -----------------------------
# SNAP TO NEAREST STREAM
# -----------------------------
if streams.crs.is_geographic:
    streams_proj = streams.to_crs(epsg=3857)
    point_for_distance = gpd.GeoSeries([point_proj], crs=streams.crs).to_crs(epsg=3857)[0]
else:
    streams_proj = streams
    point_for_distance = point_proj

streams_proj["dist"] = streams_proj.geometry.distance(point_for_distance)

start_idx = streams_proj["dist"].idxmin()
start_link = streams.loc[start_idx, link_field]

print(f"Start LINKNO: {start_link}")

# -----------------------------
# TRACE UPSTREAM
# -----------------------------
upstream_links = nx.descendants(G_rev, start_link)
upstream_links.add(start_link)

print(f"Upstream segment count: {len(upstream_links)}")

# -----------------------------
# EXTRACT SUBSETS
# -----------------------------
upstream_streams = streams[streams[link_field].isin(upstream_links)].copy()
upstream_basins = basins[basins[basin_link_field].isin(upstream_links)].copy()
upstream_basins = upstream_basins.sort_values(by=basin_link_field)

# -----------------------------
# SAVE OUTPUTS (USING ENHANCED EXPORT)
# -----------------------------
export_fixed(upstream_streams, output_streams_fp)
export_fixed(upstream_basins, output_basins_fp)

print("Done.")
print(f"Saved upstream streams → {output_streams_fp}")
print(f"Saved upstream basins  → {output_basins_fp}")

# -----------------------------
# OPTIONAL DEBUG CHECKS
# -----------------------------
missing = upstream_links - set(basins[basin_link_field])
if missing:
    print(f"Warning: {len(missing)} upstream links have no matching basin.")

Loading geofabric layers...
Streams CRS: EPSG:3979
Start LINKNO: 188266
Upstream segment count: 42547
Done.
Saved upstream streams → final_basin/Sask_streams.shp
Saved upstream basins  → final_basin/Sask_basins.shp


In [1]:
# creates 1 large basin of the entire area
# change input and output names
import geopandas as gpd

# 1. Load your basins shapefile
gdf = gpd.read_file("merged_basins/saskTotalBasins.shp")
print(f"Original number of polygons: {len(gdf)}")

# 2. Combine geometries
merged_geometry = gdf.geometry.union_all()

# --- NEW: Calculate the area ---
# .area returns meters squared if the CRS is projected in meters
area_in_sq_meters = merged_geometry.area 
area_in_sq_km = area_in_sq_meters / 1_000_000
print(f"Total Basin Area: {area_in_sq_km:,.2f} sq km")
# -------------------------------

# 3. Convert the merged geometry back into a GeoDataFrame
merged_gdf = gpd.GeoDataFrame(geometry=[merged_geometry], crs=gdf.crs)

# 4. Export the entire merged boundary to a new shapefile
output_filename = "merged_basins/saskTotalBasins_boundary.shp"
merged_gdf.to_file(output_filename)

print(f"Successfully dissolved all boundaries and saved to {output_filename}")

Original number of polygons: 13709
Total Basin Area: 83,144.80 sq km
Successfully dissolved all boundaries and saved to merged_basins/saskTotalBasins_boundary.shp


In [2]:
# trims lakes down to smaller area
# change input and output names

import geopandas as gpd

# 1. Load your lakes shapefile and your basin boundary shapefile
lakes_gdf = gpd.read_file("lakes/filtered_lakes.shp")
intitial_gdf = gpd.read_file("final_basin/Sask_basins.shp")

boundary_gdf = intitial_gdf.geometry.union_all()

# IMPORTANT: Both layers must use the exact same Coordinate Reference System (CRS)
# If they don't match, we reproject the lakes layer to match the boundary
if lakes_gdf.crs != intitial_gdf.crs:
    print("CRS mismatch detected. Reprojecting lakes layer...")
    lakes_gdf = lakes_gdf.to_crs(boundary_gdf.crs)

# 2. Clip the lakes to the boundary
# This acts as a spatial cookie-cutter
trimmed_lakes_gdf = gpd.clip(lakes_gdf, boundary_gdf)

# 3. Save the trimmed lakes to a new shapefile
output_filename = "lakes/lakes_within_boundary.shp"
trimmed_lakes_gdf.to_file(output_filename)

print(f"Original lake count: {len(lakes_gdf)}")
print(f"Trimmed lake count: {len(trimmed_lakes_gdf)}")
print(f"Saved trimmed lakes to {output_filename}")

Original lake count: 1098
Trimmed lake count: 312
Saved trimmed lakes to lakes/lakes_within_boundary.shp
